# Streaming 流式传输
流式显示也就是实时在页面更新当前模型的输出

对于基于LLM构建的应用程序，流式传输对于提升响应速度很关键，流式传输能显著提升 **用户体验（UX）** ，尤其是LLM会有推理延迟的问题

# Overview
Langchain中的 Steam 能实现的功能：
1. Stream Agent Progress - 流式传输智能体的进度，在Agent的每一个步骤之后获取状态更新
2. Stream LLM tokens - 在LLM生成时实时传输生成的tokens
3. Stream thinking/reasoning tokens - 实时传输推理或思考时产生的tokens
4. Stream custom updates - 发出用户自定义的信号
5. Stream multiple modes - 可选择多种流式传输模式
   1. updates - Agent的进度（progress）
   2. messages - LLM的tokens 和 元数据
   3. custom - 任意用户数据

# 支持的流式传输mode
将以下列出的模式的一个或多个作为列表传输给stream或astream方法里面的stream_mode参数

- updates：每次Agent执行步骤后流式传输状态更新，如果一个步骤里有多个更新，这些更新将分别传输
- messages：在调用LLM的任何节点处生成 (token, metadate) 元组
- custom：从自定义的图节点内部流式写入自定义数据

比如说, 在一次Agent单次调用Tools中，会有如下更新：
- LLM node：带有工具请求的AIMessage
- Tool node：带有执行结果的ToolMessage
- LLM node：最终响应的AIMessage

In [1]:
from langchain_openai import ChatOpenAI
import os 

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool("获取天气", description="获取当前天气")
def get_weather(city: str) -> str:
    """获取当前city的天气"""
    return f"{city}永远天晴"

agent = create_agent(
    model=model,
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "北京天气怎么样？"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': '获取天气', 'args': {'city': '北京'}, 'id': '019d0437dcb5e46f8d11794029356082'}]
step: tools
content: [{'type': 'text', 'text': '北京永远天晴'}]
step: model
content: [{'type': 'text', 'text': '北京的天气目前是晴天。'}]
